# rpart 教程（3/3）：横切专题

 >  本文件在系列中的位置 ：这是 rpart 系列教程的 第 3 部分 （末篇），承接前三个文件，讲与 method  正交 的四个横切专题。这些专题不隶属于某个 method，而是跨  `class` / `anova` / `poisson` /生存都适用。
 >
 >  系列结构 ：
 > -  `00_总览与环境.ipynb` ：预备 + 附录 A（predict 速查）+ 附录 B（常见坑）
 > -  `01_建树与剪枝机制.ipynb` ：建树与剪枝机制 + 附录 A（Breiman 排序定理证明）+ 附录 B（1-SE 深探）
 > -  `02_按method展开.ipynb` ：class / anova / poisson / 生存 + 附录 C（其余函数与数据集）
 > -  本文件（3） ：横切专题
 >
 >  前置知识 ：需要先读过  `01_建树与剪枝机制.ipynb` （cp、1-SE）与  `02_按method展开.ipynb` （各 method 的不纯度口径）。

---

## 本文件导读

四个横切专题：

1.  3.1 代价敏感 ：先验法 / 代价法 / 权重法三法等价，abalone 刻意非平衡（ >12）+ 代价矩阵（H→L 代价 5 倍）；包含原理详解、结构可视化对比及量化验证；
2.  3.2 缺失值与 surrogate ：建模时 improve 手算（排除缺失基数）、预测时按 agree 降序启用、全缺失按多数类；
3.  3.3 变量重要性 ：rpart 自带 importance 的局限（friedman1 例子）、permutation importance 对照；
4.  3.4 时间复杂度 ：分类-全类别 / 分类-全连续 / 回归三组时间实验，验证近线性于 n。

 >  本文件的主线 ：每个专题都遵循「 业务背景 → 数据勘察 → 建模/实验 → 结果判读 → 要点 」的节奏；勘察统一用  `skimr::skim_tee()`  并补派生分析。

 >  与前文的衔接 ：cp 表读法、1-SE 规则、 `prune_1se()`  的逐行解释在  `01_建树与剪枝机制.ipynb`  已讲透，本文件代码块中仍就地定义  `prune_1se()` （保持自包含），但不再逐行注释其含义。 

# 第 3 节 横切专题 

本节主题与 method 正交：代价敏感、缺失值与 surrogate、变量重要性、时间复杂度。 

## 3.1 代价敏感：先验法、代价法、权重法 

非平衡数据 + 错判代价不对称时，rpart 提供三种等价手段：

-  先验法 ： `parms = list(prior = ...)` ，按代价调整类先验；
-  代价法 ： `parms = list(loss = ...)` ，直接给代价矩阵；
-  权重法 ： `weights = ...` ，给每个样本 case weight。

 等效条件 ：先验法要求 prior 按代价矩阵调整（$\text{prior}_k \propto \pi_k \cdot \sum_j C_{kj}$）；代价法要求直接提供代价矩阵 $C_{kj}$；权重法要求 case weight 与代价成比例（$w_i \propto C_{y_i, \cdot}$）。满足时三者数学等价—— 直观地说，就是把「错判成本高」的信息，分别折算进「类先验」「损失矩阵」或「样本权重」里 。

 >  先说坑（为什么本节要把  `cp`  设为  `1e-5` ） ：本节建树时把  `cp`  设为  `1e-5`  而不是默认的  `0.01`  或绝对的  `0` 。原因有三：
 > 1.  强迫模型关注少数类 ：在极度非平衡的数据下，默认的  `cp = 0.01`  会让模型认为划分少数类带来的整体 Gini 下降“不划算”而停止分裂（直接长成单节点树）。降到  `1e-5`  是为了强迫树生长，让代价信息得以发挥作用。
 > 2.  保证三法对比的绝对公平（控制变量） ：为了验证先验法、代价法、权重法的数学等价性，必须拉齐建树门槛（同一起跑线），排除因参数不同导致的树结构差异。
 > 3.  规避浮点误差 ：如果设  `cp = 0` ，在涉及极端权重和代价矩阵时，浮点计算误差会导致边界节点的分裂判定极不稳定。设一个极小的正数  `1e-5` ，相当于加了一个“容差阈值”，屏蔽了微观噪声。这是工程实践中的最佳控制手段，最终的树深度依然会通过后续的  1-SE 剪枝  来控制。

 问题背景（刻意制造的非平衡与代价不对称） ： `02_按method展开.ipynb`  2.1.1 节我们刻意保持了数据的平衡。但在真实业务中， “老鲍鱼”的比例其实很低 ，且 漏判老鲍鱼的代价极大 （因为老鲍鱼价格昂贵，误判会造成巨大经济损失）。

 >  教学点设计：有意设计的非平衡场景 。为了演示代价敏感方法如何“拯救”被多数类淹没的少数类，我们在这里有意将阈值改为  `Rings > 12` （更老的个体），这使得少数类 H 的占比骤降至约  1/5（非平衡） 。同时，我们人为设定 把老鲍鱼误判为年轻（H→L）的代价是反过来（L→H）的 5 倍 。 此时 rpart 的关注点必须发生转变 ：不能再像第 2 部分那样纯粹追求特征空间分割的纯净度，而是要通过先验、代价或权重， 强行纠正多数类主导的惰性 ，使得模型真正去关注那些代价极高的少数类。这种“非平衡 + 代价不对称”的设计，是检验先验法、代价法、权重法三法等效性的最佳试验场。

 数据变量 ：因变量  `Rings`  按  >12 二值化为 L/H；自变量同 2.1.1。 先勘察非平衡程度，再建模 ；以下各代码块各自自包含，可单独运行。 

### 💡 三种方法的原理与参数设置

在动手建模之前，我们需要清晰理解这三种方法在  `rpart`  中的实现机制及其等效关系。它们虽然参数不同，但在满足特定条件下，最终学到的决策边界是完全一致的。

| 方法 | rpart 参数 | 核心原理 | 等效转换公式 |
| :--- | :--- | :--- | :--- |
|  先验法 (Prior)  |  `parms = list(prior = pr)`  | 修改类的先验概率，使模型在计算不纯度（Gini/Entropy）时，认为少数类的出现频率更高，从而更愿意为其分裂。 | $\text{prior}_k = \frac{\pi_k \cdot \sum_j C_{kj}}{\sum_m \pi_m \cdot \sum_j C_{mj}}$  <br> 其中 $\pi_k$ 是原始样本比例，$C_{kj}$ 是代价矩阵第 k 行之和。 |
|  代价法 (Loss)  |  `parms = list(loss = cm)`  | 直接在分裂准则中最小化期望错判损失 $\sum_k \sum_j C_{kj} p_k$，而非最小化不纯度。这是最纯粹的代价敏感形式。 | 无需转换，直接传入对称或非对称代价矩阵 $C_{kj}$（对角线通常为0）。 |
|  权重法 (Weights)  |  `weights = w`  | 给每个样本赋予 case weight，物理上放大少数类样本在节点统计量中的贡献。等价于将少数类样本“复制”多份。 | $w_i = \sum_j C_{y_i, j}$  <br> 即样本 i 的权重等于其真实类别对应行的代价之和。 |

 > ⚠️  关键提醒 ：
 > - 三种方法的 等效性依赖于参数的正确换算 。如果 prior、loss、weights 之间不满足上述转换关系，得到的树结构和预测结果将不同。
 > -  代价法 (Loss)  是最底层的实现，先验法和权重法本质上是它的两种等价变形。
 > - 在本教程中，我们设定 H→L 代价为 5，L→H 代价为 1，因此：
 >   - 权重法：$w_H = 5, w_L = 1$
 >   - 先验法：$\text{prior}_H \propto \pi_H \times 5, \quad \text{prior}_L \propto \pi_L \times 1$
 >   - 代价法：直接传入  `matrix(c(0,1,5,0), nrow=2)` 

### 🛠️ 3.1.0 环境与数据准备（公共基础）

为避免后续代码重复，本节统一完成数据加载、非平衡处理、代价矩阵定义及通用剪枝函数的声明。 注意：此处已补充三种方法的初始建树代码，以确保后续可视化代码块能正常运行。 

In [ ]:
options(warn = -1)                                  # 忽略所有警告信息，保持输出整洁
options("width" = 140)                              # 设置控制台输出宽度为140字符，防止换行过多
options(repr.plot.width = 22, repr.plot.height = 10) # 设置绘图区域的宽和高（英寸），确保图表清晰

library(rpart)                                      # 加载 rpart 包，用于构建决策树
library(AppliedPredictiveModeling)                  # 加载 AppliedPredictiveModeling 包，获取 abalone 数据集
library(skimr)                                      # 加载 skimr 包，用于快速数据概览
library(dplyr)                                      # 加载 dplyr 包，用于数据操作管道

# ---- 数据勘察与非平衡处理 ----
data(abalone)                                       # 从包中加载 abalone 数据集
d_ab_cs   <- abalone %>%                           # 开始数据处理管道，存储到 d_ab_cs
  mutate(Rings = factor(Rings > 12, labels = c("L", "H"))) # 创建新变量 Rings: 若 Ring > 12 则为 "H" (老)，否则为 "L" (年轻)

cat("=== 数据概览 ===\n")                           # 打印标题
d_ab_cs %>% skim_tee()                           # 打印数据详细摘要并返回数据框以便后续使用
cat("\n=== 类别分布 ===\n")                         # 打印标题
print(table(d_ab_cs$Rings))                      # 打印类别频数表
print(prop.table(table(d_ab_cs$Rings)))          # 打印类别比例表

# ---- 划分训练集与测试集 ----
set.seed(100)                                       # 设置随机种子，保证结果可复现
train_indices   <- sample(nrow(d_ab_cs), nrow(d_ab_cs) * 0.5) # 随机抽取50%的行索引作为训练集索引
tr_ab_cs   <- d_ab_cs[train_indices, ]           # 根据索引提取训练集
te_ab_cs   <- d_ab_cs[-train_indices, ]           # 根据负索引提取测试集

# ---- 代价矩阵定义 ----
# H->L (漏判老鲍鱼) 代价 5; L->H (误判年轻鲍鱼) 代价 1
cost_matrix   <- matrix(c(0, 1, 5, 0), nrow = 2, byrow = TRUE) # 构建2x2代价矩阵: 行代表预测值(L,H)，列代表真实值(L,H)
dimnames(cost_matrix)   <- list(predicted = levels(d_ab_cs$Rings), actual = levels(d_ab_cs$Rings)) # 为矩阵行列添加名称，方便阅读
cat("\n=== 代价矩阵 ===\n")                         # 打印标题
print(cost_matrix)                                  # 打印代价矩阵

# ---- 通用控制参数与剪枝函数 ----
control_params   <- rpart.control(cp = 1e-5)          # 设置控制参数: cp=1e-5，允许树充分生长以捕捉少数类信号

prune_1se   <- function(model_object, se = 1) {       # 定义1-SE剪枝函数
  cptab        <- model_object$cptable                # 获取模型的复杂度参数表
  xerror       <- cptab[, "xerror"]                   # 提取交叉验证误差均值列
  xstd         <- cptab[, "xstd"]                     # 提取交叉验证误差标准差列
  cp_values    <- cptab[, "CP"]                       # 提取复杂度参数 CP 列
  min_idx      <- which.min(xerror)                   # 找到最小交叉验证误差对应的索引
  threshold    <- xerror[min_idx] + se * xstd[min_idx] # 计算1-SE阈值: 最小误差 + 1倍标准差
  cp_best      <- max(cp_values[xerror <= threshold])  # 在误差小于阈值的范围内，选择最大的CP值（最简树）
  rpart::prune(model_object, cp = cp_best)          # 使用选定的CP值对树进行剪枝并返回
}

# 预计算权重与调整先验（供后续直接使用）
prop_L   <- mean(tr_ab_cs$Rings == "L")             # 计算训练集中 L 类的比例
adjusted_prior   <- c(prop_L, (1-prop_L) * 5)           # 计算调整后的先验概率分子: L类比例 1, H类比例 5
adjusted_prior   <- adjusted_prior / sum(adjusted_prior) # 归一化先验概率
sample_weights   <- ifelse(tr_ab_cs$Rings == "H", 5, 1) # 计算样本权重: H类权重为5, L类权重为1

# ============================================
# 【修复】在此处预先建立三个 Full Tree 模型
# 解决后续 3.1.1 中 tree_xxx_full 未找到的问题
# ============================================
set.seed(22)                                        # 设置随机种子
m_cs_prior_full   <- rpart(Rings ~ ., data = tr_ab_cs, parms = list(prior = adjusted_prior), control = control_params) # 使用先验法建树
m_cs_loss_full    <- rpart(Rings ~ ., data = tr_ab_cs, parms = list(loss = cost_matrix),      control = control_params) # 使用代价法建树
m_cs_weight_full   <- rpart(Rings ~ ., data = tr_ab_cs, weights = sample_weights,             control = control_params) # 使用权重法建树 

### 🖼️ 3.1.1 决策树结构可视化对比

以下代码将三种方法生成的完整树进行适当剪枝后，以最清晰的辐射状树形图逐一输出。我们通过调整  `tweak`  参数将节点拉宽，确保文字不重叠。请重点对比三张图的分支结构与叶子节点类别。 

In [ ]:
library(rpart.plot)                                 # 加载 rpart.plot 包，用于绘制精美的决策树
options(repr.plot.width = 18, repr.plot.height = 10) # 调整绘图区域大小

# 辅助函数：剪枝到适合展示的深度（取 cp 表第 5 行）
get_light_cp   <- function(tree_model) {              # 定义获取轻度剪枝CP值的函数
  idx   <- min(5, nrow(tree_model$cptable))           # 取第5行或最后一行（如果不足5行），避免越界
  return(tree_model$cptable[idx, "CP"])             # 返回该行的CP值
}

# 基于 3.1.0 中已建立的 full tree 进行轻度剪枝
m_cs_prior_light    <- prune(m_cs_prior_full, cp = get_light_cp(m_cs_prior_full))   # 对先验法树进行轻度剪枝
m_cs_loss_light     <- prune(m_cs_loss_full, cp = get_light_cp(m_cs_loss_full))     # 对代价法树进行轻度剪枝
m_cs_weight_light   <- prune(m_cs_weight_full, cp = get_light_cp(m_cs_weight_full)) # 对权重法树进行轻度剪枝

# 统一绘图参数
plot_params   <- list(
  type = 1, extra = 104, branch = 1, branch.lty = 1,  # type=1标签式, extra=104显示概率和类别, branch=直角分支
  shadow.col = "gray", box.palette = "GnBu",          # 阴影灰色, 盒子颜色蓝绿色系
  fallen.leaves = TRUE, tweak = 1.2, yesno = 2, split.cex = 1.1 # 叶子底部对齐, 字体放大1.2倍, 显示Yes/No, 分裂文字大小
)


do.call(rpart.plot, c(list(m_cs_prior_light, main = "先验法 (Prior)"), plot_params)) # 绘制先验法树


do.call(rpart.plot, c(list(m_cs_loss_light, main = "代价法 (Loss)"), plot_params))   # 绘制代价法树


do.call(rpart.plot, c(list(m_cs_weight_light, main = "权重法 (Weights)"), plot_params)) # 绘制权重法树 

### 💡 核心洞察：为何结构相同但概率不同？

当你依次查看这三张图时，可能会产生两个疑问。以下是基于底层数学机制的深度解析：

#### 1. 分支结构与最终类别：完全一致 ✅
经过仔细对比，这三棵树的 拓扑结构、分裂变量及阈值完全一致 （根节点均为  `ShellWeight < 0.18` ，后续所有分裂点及叶子节点位置均严丝合缝对应）。更重要的是，最底层叶子节点的 预测类别（第一行字母）序列也完全相同 （均为  `L, L, L, H, L, H, H, H, H` ）。

这完美印证了教程的核心观点：**先验法、代价法、权重法在满足等效条件时，构建出的决策边界和硬分类结果是数学等价的。 它们只是在实现路径上有所不同，但最终都成功“纠正”了多数类主导的惰性，找到了同一个代价最优解。

#### 2. 节点概率分布：为何存在显著差异？⚠️
虽然结构和类别相同，但节点内部显示的 概率分布（第二行数字）**却有明显区别（尤其是代价法与其他两法）。这并非错误，而是由  `rpart`  底层计算“不纯度”和“节点统计量”的机制不同导致的：

| 方法 | 节点内概率的计算逻辑 | 直观理解 |
| :--- | :--- | :--- |
|  先验法 (Prior)  | $\frac{n_k \times \text{prior}_k}{\sum_j n_j \times \text{prior}_j}$ | 把原始样本数乘以调整后的先验概率。 它模拟的是“如果总体分布如先验所示，各类别的期望占比” 。 |
|  权重法 (Weights)  | $\frac{\sum_{i \in k} w_i}{\sum_{all} w_i}$ | 直接使用 case weight 求和。 因为 $w_H=5, w_L=1$，所以 H 类样本在统计上被“放大”了 5 倍 。这与先验法在数学上是等价的（当 prior 按代价归一化时）。 |
|  代价法 (Loss)  |  基于风险最小化的隐式概率  |  这是最特殊的一个 。代价法不直接修改样本计数，而是在分裂准则中直接最小化 $\sum C_{kj} p_k$。因此，节点内显示的往往是 原始样本比例 ，或者是在特定 loss 函数下的 风险等价概率 ，而非简单的加权频率。例如根节点显示  `H .83 .17` ，反映的是在当前 Loss 定义下 H 类的“有效影响力”占据了 83%，尽管其原始样本占比可能远低于此。 |

 > 💡  工程实践建议 ：
 > -  看结构和叶子类别 ：三者等价，任选其一均可获得相同的最优决策边界。
 > -  看节点数字 ：先验法 = 权重法 ≠ 代价法。若需可解释的概率输出或用于后续风险评分，推荐使用 权重法 或 先验法 ；若只关心最小化业务损失且不需要概率校准， 代价法 是最纯粹的数学表达。
 > -  软概率需谨慎 ：代价法输出的概率可能需要额外的后处理（Post-hoc Calibration）才能用于排序或阈值调整，因为它原生优化的是 Loss 而非概率估计。 

### 📊 3.1.3 测试集代价评估（量化验证）

以下代码将上述生成的三个轻度剪枝树模型应用到测试集  `te_ab_cs`  上，并计算各自的加权总代价。这将进一步从量化角度验证三法在决策层面的等价性。 

In [ ]:
# 1. 先验法模型在测试集上的代价
p_cls_prior   <- predict(m_cs_prior_light, te_ab_cs, type = "class") # 使用先验法模型预测测试集类别
conf_prior   <- table(te_ab_cs$Rings, p_cls_prior)                   # 生成混淆矩阵: 行为真实值, 列为预测值
err_cost_prior   <- sum(conf_prior * cost_matrix)                        # 计算加权总代价: 混淆矩阵元素乘以对应代价矩阵元素后求和

# 2. 代价法模型在测试集上的代价
p_cls_loss   <- predict(m_cs_loss_light, te_ab_cs, type = "class")   # 使用代价法模型预测测试集类别
conf_loss   <- table(te_ab_cs$Rings, p_cls_loss)                     # 生成混淆矩阵
err_cost_loss   <- sum(conf_loss * cost_matrix)                          # 计算加权总代价

# 3. 权重法模型在测试集上的代价
p_cls_weight   <- predict(m_cs_weight_light, te_ab_cs, type = "class") # 使用权重法模型预测测试集类别
conf_weight   <- table(te_ab_cs$Rings, p_cls_weight)                 # 生成混淆矩阵
err_cost_weight   <- sum(conf_weight * cost_matrix)                      # 计算加权总代价

# 输出结果对比
cat("=== 测试集加权总代价对比 ===\n")                 # 打印标题
cat(sprintf("先验法 (Prior)   总代价: %d\n", err_cost_prior))   # 格式化输出先验法总代价
cat(sprintf("代价法 (Loss)    总代价: %d\n", err_cost_loss))    # 格式化输出代价法总代价
cat(sprintf("权重法 (Weights) 总代价: %d\n", err_cost_weight)) # 格式化输出权重法总代价

# 可选：打印混淆矩阵以便详细查看
cat("\n--- 先验法混淆矩阵 ---\n"); print(conf_prior)  # 打印先验法混淆矩阵
cat("\n--- 代价法混淆矩阵 ---\n"); print(conf_loss)   # 打印代价法混淆矩阵
cat("\n--- 权重法混淆矩阵 ---\n"); print(conf_weight) # 打印权重法混淆矩阵 

 >  本节要点 
 > 1. 代价敏感建模前先勘察非平衡程度与代价方向，确认少数类规模与代价矩阵一致。
 > 2. 先验法、代价法、权重法在等效条件下结果一致；权重法最直观。
 > 3. 代价敏感场景避免  `cp = 0` ，用  `cp = 1e-5`  更稳定。
 > 4. 评估用「测试集总损失」（代价矩阵加权），而非单纯错误率；三法结果宜并列对比。 

## 3.2 缺失值与 surrogate 

rpart 的两阶段缺失处理：
-  建模时 ：主分割变量缺失的样本不参与该分割的 improve 计算；同时搜索 surrogate splits（与主分割一致性最高的其他变量）。
-  预测时 ：主分割变量缺失 → 按  `agree`  降序用 surrogate；全缺失 → 按  `usesurrogate`  决定。

 术语先明（本节反复用到，先在此一次讲清） ：

-  主分割（primary split）/ 主分割变量 ：rpart 在当前节点选出的、使不纯度下降最大的那个切分规则，以及它用到的变量。例：根节点主分割是  `ShellWeight < 0.2545` ， ShellWeight 就是主分割变量 ，这条规则就是主分割。
-  代理分割（surrogate split） ：用 其他变量 构造一个切分，尽量 复现主分割把样本分到左/右哪一侧 。它不是独立找最优，而是**“模仿主分割” 。
-  agree（一致性） ：代理分割与主分割在训练样本上 把样本分到同一侧的比例**，取值 0~1。agree 越高，说明这个代理越能**“顶替” 主分割。 `summary()`  里 surrogate 表就按 agree 从高到低排序。 agree 是一个单值 ：它衡量的是两次劈法（主分割、代理分割）把同一批样本送进同一侧的比例，不分左右分别评价——左右是同一件事的两面，一次二分支只有一种 “分法” 。

一句话： “主分割一致性最高的其他变量” = agree 最大的代理变量**，它是主分割变量缺失时预测阶段的第一顺位替补。

 问题背景（缺失场景） ：仍用鲍鱼年龄分类问题。现实中测量可能失败或漏记——比如某只鲍鱼的壳重（ShellWeight）没测到，但其他尺寸重量都齐全。rpart 的 surrogate 机制正是为了在 主分割变量缺失时，用与之最相关的其他变量顶替 ，而不是简单丢掉这条样本。本节演示缺失对建模和预测的影响。 

### 3.2.1 建模时缺失：improve 的手算对照 

将训练集第一行 ShellWeight 置为缺失，用  `maxdepth = 1`  只看根节点分割。 先勘察缺失注入后的数据，再建模与手算 ；两个代码块各自自包含，独立可运行。

 >注意这里  `maxdepth = 1` ，树只有根节点一层： 根节点的主分割就是  `ShellWeight < 0.2545` ，主分割变量就是 ShellWeight 。下面手算 improve 时， “父节点基数”要用 排除 ShellWeight 缺失样本之后 的样本数（本例 2087），这正是 rpart 建模时的口径——缺失样本不参与该分割的 improve 计算。 

以下代码 向数据中人为注入缺失值，并查看其对统计摘要和频数分布的影响 。代码将第一行数据的  `ShellWeight`  设为  `NA` ，随后使用  `skim_tee()`  直接查看了缺失情况，并用  `table(..., exclude = NULL)`  统计了含缺失值时的频数，为后续的模型建模及手算验证做铺垫。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(AppliedPredictiveModeling)                # 加载数据包
library(skimr); library(dplyr)                    # 加载 skimr 与 dplyr
data(abalone); d_ab_raw   <- abalone          # 载入数据并存储到 d_ab_raw
d_ab_raw$Rings   <- factor(d_ab_raw$Rings > 9, labels = c("L", "H")) # Rings 二值化（平衡）
set.seed(100); train_idx   <- sample(nrow(d_ab_raw), nrow(d_ab_raw) * 0.5) # 随机抽一半作训练集
tr_ab_raw   <- d_ab_raw[train_idx, ]     # 训练集
tr_ab_miss   <- tr_ab_raw              # 复制训练集用于注入缺失
tr_ab_miss[1, "ShellWeight"]   <- NA        # 将第一行 ShellWeight 置为缺失

tr_ab_miss %>% skim_tee()                 # 打印全表摘要并返回原数据（n_missing 直接显示缺失）
with(tr_ab_miss, table(cut(ShellWeight, c(0, 0.2545, Inf)), # 主分割变量分箱计数
                 Rings, exclude = NULL))          # 含缺失（exclude = NULL 保留 NA）
nrow(tr_ab_miss) - sum(is.na(tr_ab_miss$ShellWeight)) # 排除缺失后的根节点基数（skim 不直接给这个派生量） 

以下代码 验证含缺失值时，rpart 是如何计算分割优度（improve）的 。代码在受限深度下建树并提取节点摘要，然后手工按**“排除缺失后” 的样本基数重新计算了 Gini 下降量，证明 rpart 内部的缺失处理机制是自动忽略缺失样本的。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(rpart); library(AppliedPredictiveModeling) # 加载 rpart 与数据包
data(abalone); d_ab_raw   <- abalone          # 载入数据并存储到 d_ab_raw
d_ab_raw$Rings   <- factor(d_ab_raw$Rings > 9, labels = c("L", "H")) # Rings 二值化（平衡）
set.seed(100); train_idx   <- sample(nrow(d_ab_raw), nrow(d_ab_raw) * 0.5) # 随机抽一半作训练集
tr_ab_raw   <- d_ab_raw[train_idx, ]     # 训练集
tr_ab_miss   <- tr_ab_raw              # 复制训练集用于注入缺失
tr_ab_miss[1, "ShellWeight"]   <- NA        # 将第一行 ShellWeight 置为缺失
with(tr_ab_miss, table(cut(ShellWeight, c(0, 0.2545, Inf)), # 主分割变量分箱计数
                 Rings, exclude = NULL))          # 含缺失

set.seed(2); m_miss   <- rpart(Rings ~ ., data = tr_ab_miss, # 建分类树（仅根节点）
                           method = "class", maxdepth = 1) # 限制深度为 1
summary(m_miss)                                     # 查看根节点分割与 improve

gini   <- function(n1, n2) 1 - (n1/(n1+n2))^2 - (n2/(n1+n2))^2 # Gini 不纯度函数
imp_manual   <- 2087 * gini(1044,1043) -                   # 父节点不纯度（排除缺失后 2087 例）
       1166 * gini(851,315) -                     # 左子节点不纯度
       921 * gini(192,729)                        # 右子节点不纯度
cat("手算 improve = ", round(imp_manual, 4), "\n")       # 输出手算 improve 

 手算（排除 1 例缺失，共 2087 例）： 

 `
improve = 2087·G(1044,1043) − 1166·G(851,315) − 921·G(192,729)
` 

注意：rpart 的做法是父节点 improve 用 排除缺失后 的样本数（2087），左右子节点数也相应为 1166/921（而非 1166/922）。按此口径手算与 summary 一致。 

### 3.2.2 预测时缺失：surrogate 按序使用 

用  `maxsurrogate = 7`  让 rpart 搜索全部代理分割， `maxdepth = 1`  便于观察。 先勘察代理搜索所需的一致性信息，再建模 ；两个代码块各自自包含，独立可运行。 

以下代码 在建模前探查变量之间的相关性**。代码打印了全表摘要，并计算了自变量之间的相关矩阵以及目标主分割变量  `ShellWeight`  与其他变量的相关系数。这有助于预判哪些变量最有可能成为高质量的代理分割（surrogate split）。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(AppliedPredictiveModeling)                # 加载数据包
library(skimr); library(dplyr)                    # 加载 skimr 与 dplyr
data(abalone); d_ab_raw   <- abalone          # 载入数据并存储到 d_ab_raw
d_ab_raw$Rings   <- factor(d_ab_raw$Rings > 9, labels = c("L", "H")) # Rings 二值化（平衡）
set.seed(100); train_idx   <- sample(nrow(d_ab_raw), nrow(d_ab_raw) * 0.5) # 随机抽一半作训练集
tr_ab_sur   <- d_ab_raw[train_idx, ]     # 划分训练集
te_ab_sur   <- d_ab_raw[-train_idx, ]     # 划分测试集

tr_ab_sur %>% skim_tee()                     # 打印全表摘要并返回原数据
numeric_vars   <- names(tr_ab_sur)[sapply(tr_ab_sur, is.numeric)] # 数值型变量名
cor(tr_ab_sur[, numeric_vars])               # 数值变量相关矩阵（skim 不做）
cor(tr_ab_sur$ShellWeight, tr_ab_sur[, numeric_vars]) # ShellWeight 与各变量的相关性 

 判读要点 ：与 ShellWeight 相关性最高的变量最可能成为 surrogate，相关性排序应与后面 summary 的 surrogate 顺序大致对应。鲍鱼的各部位重量与尺寸高度相关，所以 surrogate 的效果通常不错。 

以下代码 演示 rpart 是如何寻找并排序代理分割（Surrogate Splits）的 。代码允许搜索所有潜在的代理变量，通过  `summary()`  输出其一致性排序。随后，代码手写了循环逻辑，逐一计算每个数值变量与主分割的一致性（即 agree），手工复现了代理变量的搜索过程并与软件结果进行对照。

 agree 的定义与手算对应 （对照下面代码逐行理解）：
-  `main_left <- tr_ab_sur$ShellWeight < 0.2545` ：主分割给出的 *“左支指示” ，每个样本 TRUE/FALSE。
- 对候选变量  `v`  和候选切点  `c` ： `(x < c)`  是该代理分割给出的 “左支指示” 。
-  `mean((x < c) == main_left)` ：两个切分 把样本分到同一侧的比例**，这就是 agree。
- 对每个变量取 agree 最大的切点作为它的最优代理；最后按 agree 降序排列，即得 surrogate 优先级。

 为什么 agree 不分左右分别评价 ：一次二分支就是把 同一批样本 劈成两堆（左/右只是这两堆的名字），所以**“左右是同一件事的两面” ，不存在两个独立的侧面。
-  单值本质 ： `mean((x < c) == main_left)`  对所有样本只求一次平均，得到一个 0~1 的标量。左支的相符率定了，右支的相符率就是它的补（同一批样本、同一个布尔比较），分别报告是冗余的。
-  交叉表视角 ：把主分割与代理分割的左右归属做成 2×2 表，agree = 对角线和 / 总数。它天然把 “主左配代理左” 和 “主右配代理右” 合在一起算，没有 “先算左再算右” 的分步。
-  反例 ：某代理在左支 100 例全对、右支 100 例全错。若分左右评价，会得到 “左 1.0、右 0.0” 两个矛盾数字，无法决策；合成单值 agree = 0.5，含义清晰——整体跟抛硬币差不多，不该启用。rpart 要的正是能直接排序、直接定优先级的单指标。

 最小可手算实例（8 个样本） ：设根节点主分割为  `ShellWeight < 0.2545` ，候选代理变量  `WholeWeight`  取切点  `c = 0.325` 。

| 分割 | 左子集 | 右子集 |
|---|---|---|
|  主分割  | {1, 2, 3, 4} | {5, 6, 7, 8} |
|  代理分割 （ `WholeWeight < 0.325` ） | {1, 2, 3} | {4, 5, 6, 7, 8} |

 同向配对后数一致 ：主左 {1,2,3,4} ↔ 代左 {1,2,3}，共同在左的是  {1,2,3} （3 个）；主右 {5,6,7,8} ↔ 代右 {4,5,6,7,8}，共同在右的是  {5,6,7,8} （4 个）。 agree = (3 + 4) / 8 = 7/8 = 0.875 。

 用 2×2 表看更直接 ：

| | 代理左 {1,2,3} | 代理右 {4,5,6,7,8} |
|---|---|---|
|  主左 {1,2,3,4}  | 3（1,2,3） | 1（4） |
|  主右 {5,6,7,8}  | 0 | 4（5,6,7,8） |

agree = 对角线和 / 总数 = (3 + 4) / 8 =  0.875 。

 一句话 ：主分割两堆 {1,2,3,4} / {5,6,7,8}，代理分割两堆 {1,2,3} / {4,5,6,7,8}，同向配对后， 同堆样本 3 + 4 = 7 个，agree = 7/8 。

 分工 ：agree 只在 “代理像不像主分割” 这一层做单值评价，负责 排优先级**；左右方向的区分只发生在两个别的环节——(1) improve 计算时父节点不纯度减去左、右子节点不纯度（左右子各自算）；(2) 预测时用代理切分 给某个具体样本定侧 （落到某一侧，但用的仍是同一个 agree 排序）。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(rpart); library(AppliedPredictiveModeling) # 加载 rpart 与数据包
data(abalone); d_ab_raw   <- abalone          # 载入数据并存储到 d_ab_raw
d_ab_raw$Rings   <- factor(d_ab_raw$Rings > 9, labels = c("L", "H")) # Rings 二值化（平衡）
set.seed(100); train_idx   <- sample(nrow(d_ab_raw), nrow(d_ab_raw) * 0.5) # 随机抽一半作训练集
tr_ab_sur   <- d_ab_raw[train_idx, ]     # 划分训练集
te_ab_sur   <- d_ab_raw[-train_idx, ]     # 划分测试集

set.seed(2)                                       # 固定种子
m_sur   <- rpart(Rings ~ ., data = tr_ab_sur,            # 建分类树（仅根节点）
              control = list(maxsurrogate = 7),   # 搜索全部代理分割
              method = "class", maxdepth = 1)     # 分类树，深度 1
summary(m_sur)                                     # 查看 surrogate 的 agree 排序

# ---- 代理搜索算法（就地演示） ----
main_split_left   <- tr_ab_sur$ShellWeight  < 0.2545           # 主分割左支指示
numeric_vars   <- names(tr_ab_sur)[sapply(tr_ab_sur, is.numeric)] # 数值型变量名
surrogate_results   <- lapply(numeric_vars, function(var_name) {             # 对每个数值变量
  x_vals   <- tr_ab_sur[[var_name]]; unique_x   <- sort(unique(x_vals))          # 变量取值与唯一值
  candidate_cuts   <- (unique_x[-1] + unique_x[-length(unique_x)]) / 2          # 相邻唯一值中点作为候选切点
  agreement_scores   <- sapply(candidate_cuts, function(cut_point) mean((x_vals  < cut_point) == main_split_left)) # 各切点与主分割的一致率（agree）
  data.frame(variable = var_name,                        # 变量名
             best_cut = candidate_cuts[which.max(agreement_scores)],          # 最优切点
             max_agree = max(agreement_scores))                    # 最大一致率
})
surrogate_df   <- do.call(rbind, surrogate_results)                        # 合并各变量结果
surrogate_df[order(-surrogate_df$max_agree), ]                          # 按一致率降序输出 

与  `summary(m_sur)`  的 Surrogate splits 顺序一致（WholeWeight → Diameter → LongestShell → …）。

 为什么 agree 决定启用顺序 ：预测时若主分割变量  `ShellWeight`  缺失，就轮到代理分割出手。rpart 按 agree 从高到低依次尝试——agree 越高，代理切分与主分割在训练集上判定越一致，顶替时把样本分错侧的概率越小。所以 WholeWeight 排第一，是因为它与  `ShellWeight < 0.2545`  的 agree 最高，是最 *“像” 主分割的替补；若 WholeWeight 也缺失，才退到 agree 次高的 Diameter，以此类推。注意这里 agree 只负责 排优先级**；真正给样本定左/右，是由被启用的代理切分当场完成的。

 观察 surrogate 的逐级启用与累积缺失误差 （ 本块自带数据与建模，独立可运行；勘察见上一块 ）： 

以下代码 演示 rpart 在预测阶段如何处理缺失值 。首先对单条样本逐步增加缺失变量，观察模型如何按代理变量的优先级（一致性从高到低）自动向下兼容进行预测。接着通过循环使整个测试集逐渐全缺失，展示当所有特征都缺失时，模型退化为按多数类预测，以及误差的累积过程。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(rpart); library(AppliedPredictiveModeling) # 加载 rpart 与数据包
data(abalone); d_ab   <- abalone                      # 载入数据并复制
d_ab$Rings   <- factor(d_ab$Rings > 9, labels = c("L", "H")) # Rings 二值化（平衡）
set.seed(100); i   <- sample(nrow(d_ab), nrow(d_ab)*0.5) # 随机抽一半作训练集
tr_ab   <- d_ab[i, ]; te_ab   <- d_ab[-i, ]               # 划分训练集与测试集
set.seed(2)                                       # 固定种子
m_sur_pred   <- rpart(Rings ~ ., data = tr_ab,            # 建分类树（仅根节点）
              control = list(maxsurrogate = 7),   # 搜索全部代理分割
              method = "class", maxdepth = 1)     # 分类树，深度 1

te_s   <- te_ab                                     # 复制测试集
te_s[1, "ShellWeight"]   <- NA                      # 缺失 ShellWeight
predict(m_sur_pred, te_s[1, ], type = "class")          # 用 WholeWeight 代理
te_s[1, "WholeWeight"]   <- NA                      # 再缺失 WholeWeight
predict(m_sur_pred, te_s[1, ], type = "class")          # 退到 Diameter
te_s[1, "Diameter"]   <- NA                         # 再缺失 Diameter
predict(m_sur_pred, te_s[1, ], type = "class")          # 退到 LongestShell

vars   <- c("ShellWeight", "WholeWeight", "Diameter", "LongestShell", # 缺失累积顺序
              "VisceraWeight", "ShuckedWeight", "Height", "Type")     # 依次缺失的变量
errs   <- numeric(length(vars)); te_s   <- te_ab        # 误差向量与重置测试集
for (k in seq_along(vars)) {                      # 逐个累积缺失
  # 关键修复：使用 NA_real  保持数值型，或根据变量类型动态选择 NA 类型
  # 这里简单起见，对于数值型变量使用 NA_real_，对于因子/字符使用 NA_character_
  if (is.numeric(te_s[[vars[k]]])) {
    te_s[[vars[k]]]   <- NA_real_
  } else {
    te_s[[vars[k]]]   <- NA_character_
  }
  errs[k]   <- mean(predict(m_sur_pred, te_s, type = "class") != te_s$Rings) # 累积缺失下的测试误差
}
data.frame(cum_missing = vars, test_error = round(errs, 4)) # 缺失累积与误差对照 

单条样本那三行  `predict`  正是**“按 agree 降序启用” 的现场演示：先缺  `ShellWeight` （agree 最高的 WholeWeight 顶替）→ 再缺  `WholeWeight` （退到 agree 次高的 Diameter）→ 再缺  `Diameter` （退到 LongestShell）。最后一行（全部自变量缺失）的误差 = 多数类比例（L 类占比），验证  `usesurrogate = 2`  的「全缺失按多数类」行为。 

 >  本节要点 
 > 1. 术语先分清： 主分割 是不纯度下降最大的切分， 代理分割 是用其他变量模仿它， agree  是代理与主分割在训练集上分到同一侧的比例（0~1，单值、不分左右分别评价，越高越优先启用）。
 > 2. 缺失处理前先勘察缺失位置、缺失对根节点基数的影响，以及代理候选变量与主分割变量的相关性。
 > 3. 建模时缺失样本不参与 improve 计算；手算需用排除缺失后的基数。
 > 4. 预测时按  `agree`  降序用 surrogate； `maxsurrogate`  控制搜索数量， `usesurrogate`  控制全缺失行为。
 > 5. 全缺失时  `usesurrogate = 2`  按多数类返回，误差 = 多数类比例。 

## 3.3 变量重要性：rpart 自带 vs permutation 

### 3.3.1 rpart 自带 importance 

定义：变量作为 主变量 的每次分割的 improve 之和，加上作为 代理变量 的 improve × adjusted agreement。输出缩放到总和 100，省略  <1% 的变量。

 问题背景（重要性解读） ：仍用鲍鱼年龄二分类问题。业务上想知道： 七个测量指标里，哪些对区分「老/年轻」最关键？  自带 importance 给出一个基于训练集分割 improve 的排序，但它对过拟合与代理使用敏感。本节先看它，再用 friedman1 展示其局限，最后用 permutation 重要性对照。

 先勘察 abalone 分类口径的数据，再建模 ；两个代码块各自自包含，独立可运行。 

以下代码 在计算变量重要性前进行快速数据勘察**。通过  `skim_tee()`  获取整体摘要，并通过  `table`  与  `prop.table`  输出目标变量的平衡度，确认数据的健康状态。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(AppliedPredictiveModeling)                # 加载数据包
library(skimr); library(dplyr)                    # 加载 skimr 与 dplyr
data(abalone); d_ab_imp   <- abalone          # 载入数据并存储到 d_ab_imp
d_ab_imp$Rings   <- factor(d_ab_imp$Rings > 9, labels = c("L", "H")) # Rings 二值化（平衡）
d_ab_imp %>% skim_tee()                     # 打印全表摘要并返回原数据
table(d_ab_imp$Rings)                       # 输出两类频数
prop.table(table(d_ab_imp$Rings))           # 输出两类比例
table(d_ab_imp$Type)                        # 类别自变量完整水平分布 

以下代码 展示 rpart 模型自带变量重要性（ `$variable.importance` ）的快速提取方法 。代码建树后，直接调用  `barplot()`  将该指标可视化为条形图。该指标主要基于训练集上的分割优度（improve）累加而成，能快速给出一版特征排名。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(rpart); library(AppliedPredictiveModeling) # 加载 rpart 与数据包
data(abalone); d_ab_imp   <- abalone          # 载入数据并存储到 d_ab_imp
d_ab_imp$Rings   <- factor(d_ab_imp$Rings > 9, labels = c("L", "H")) # Rings 二值化（平衡）
set.seed(100); train_idx   <- sample(nrow(d_ab_imp), nrow(d_ab_imp)*0.5) # 随机抽一半作训练集
tr_ab_imp   <- d_ab_imp[train_idx, ]     # 训练集
set.seed(2); m_imp_ab   <- rpart(Rings ~ ., data = tr_ab_imp, cp = 0) # 建分类树，充分生长
barplot(m_imp_ab$variable.importance,   # 画自带变量重要性
        main = "rpart 自带 importance（abalone 分类树）") # 标题 

### 3.3.2 自带的局限：friedman1 例子 

friedman1 数据：$Y = 10\sin(\pi X_1 X_2) + 20(X_3-0.5)^2 + 10X_4 + 5X_5 + \epsilon$，只有 $X_1$–$X_5$ 与 Y 相关，$X_6, X_7$ 无关。这是 纯模拟数据 ，生成机制已知，用来检验 importance 能否正确识别无关变量。 先勘察生成的数据，再建模 ；两个代码块各自自包含，独立可运行。 

以下代码 生成带有明确已知相关变量和无关变量的模拟数据（friedman1） ，并进行勘察。通过  `skim_tee()`  查看数据全貌，利用  `cor()`  计算各自变量与目标 y 的边际相关性，作为后续判断变量重要性是否准确的基准。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(skimr); library(dplyr)                    # 加载 skimr 与 dplyr
gen_friedman1   <- function(n_samples, n_features = 10, noise_std = 0.1) { # friedman1 数据生成函数
  X_matrix   <- matrix(runif(n_samples * n_features), n_samples, n_features)               # 生成 [0,1] 均匀自变量
  y_vector   <- 10 * sin(pi * X_matrix[,1] * X_matrix[,2]) +             # 交互项
       20 * (X_matrix[,3] - 0.5)^2 +                     # 二次项
       10 * X_matrix[,4] + 5 * X_matrix[,5] +                   # 线性项
       noise_std * rnorm(n_samples)                           # 噪声
  df_result   <- as.data.frame(X_matrix); names(df_result)   <- paste0("X", 1:n_features); df_result$y   <- y_vector # 组装数据框
  df_result                                              # 返回
}
set.seed(943); d_fried   <- gen_friedman1(1000, n_features = 7, noise_std = 0.1) # 生成 7 个自变量的数据

d_fried %>% skim_tee()                        # 打印全表摘要并返回原数据
cor(d_fried[, 1:7], d_fried$y)            # 各自变量与 y 的边际相关性（skim 不做） 

 判读要点 ：边际相关只能反映线性/单调信号，X1、X2 的交互项与 X3 的二次项在边际相关上可能偏弱，但它们确实与 y 有关——这正是后面 permutation importance 应识别、而自带 importance 可能失真的地方。 

以下代码 在模拟数据上暴露自带  `variable.importance`  的局限性 。由于该指标仅基于训练集的不纯度下降，它容易受到过拟合和噪声的影响。代码建树并剪枝后绘制重要性图，可以看到理论上应该毫无关联的变量  `X6` 、 `X7` ，却依然被赋予了一定的重要性数值。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(rpart)                                    # 加载 rpart 包
# ---- friedman1 数据生成函数就地定义 ----
gen_friedman1   <- function(n_samples, n_features = 10, noise_std = 0.1) { # friedman1 数据生成函数
  X_matrix   <- matrix(runif(n_samples * n_features), n_samples, n_features)               # 生成 [0,1] 均匀自变量
  y_vector   <- 10 * sin(pi * X_matrix[,1] * X_matrix[,2]) +             # 交互项
       20 * (X_matrix[,3] - 0.5)^2 +                     # 二次项
       10 * X_matrix[,4] + 5 * X_matrix[,5] +                   # 线性项
       noise_std * rnorm(n_samples)                           # 噪声
  df_result   <- as.data.frame(X_matrix); names(df_result)   <- paste0("X", 1:n_features); df_result$y   <- y_vector # 组装数据框
  df_result                                              # 返回
}
# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(model_object, se = 1) {           # 1-SE 剪枝函数
  cptab   <- model_object$cptable                         # 取出 cp 表
  xerror   <- cptab[, "xerror"]                     # 交叉验证误差均值列
  xstd   <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp_values   <- cptab[, "CP"]                             # 复杂度参数列
  min_idx   <- which.min(xerror)                    # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx] # 1-SE 阈值
  cp_best   <- max(cp_values[xerror <= threshold])         # 阈值内取最大 cp
  rpart::prune(model_object, cp = cp_best)              # 按 cp 剪枝返回子树
}
set.seed(943); d_fried   <- gen_friedman1(1000, n_features = 7, noise_std = 0.1) # 生成数据
set.seed(100); train_idx_fr   <- sample(nrow(d_fried), nrow(d_fried)*0.5) # 随机抽一半作训练集
tr_fried   <- d_fried[train_idx_fr, ]; te_fried   <- d_fried[-train_idx_fr, ]               # 划分训练集与测试集
set.seed(2); m_fried   <- rpart(y ~ ., data = tr_fried, cp = 0) # 建回归树，充分生长
m_fried_best   <- prune_1se(m_fried, se = 1)                # 1-SE 剪枝
barplot(m_fried_best$variable.importance,               # 画自带变量重要性
        main = "rpart 自带 importance（friedman1）") # 标题 

可见自带 importance 未能正确反映 X6、X7 与 Y 无关（它们仍有一定数值）。原因是自带 importance 基于「训练集上的分割 improve」，对过拟合与代理使用敏感。 

### 3.3.3 permutation 重要性 

原理：对每个变量随机置换其取值，观察预测误差的上升量；上升越多越重要。 本块自带数据、建模与置换重要性 ，独立可运行（数据勘察见 3.3.2，口径相同）。 

以下代码 使用基于预测误差的  `permutation`  重要性来对照验证 。 `vip::vi()`  会在测试数据上对每个变量进行随机置换，并通过观察 RMSE 的恶化程度来衡量其重要性。从生成的 ggplot 图中可以验证，该方法能正确地将无关变量  `X6` 、 `X7`  排在末位，其重要性几乎为零。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(rpart); library(vip); library(ggplot2)    # 加载 rpart、vip、ggplot2
# ---- friedman1 数据生成函数就地定义 ----
gen_friedman1   <- function(n_samples, n_features = 10, noise_std = 0.1) { # friedman1 数据生成函数
  X_matrix   <- matrix(runif(n_samples * n_features), n_samples, n_features)               # 生成 [0,1] 均匀自变量
  y_vector   <- 10 * sin(pi * X_matrix[,1] * X_matrix[,2]) +             # 交互项
       20 * (X_matrix[,3] - 0.5)^2 +                     # 二次项
       10 * X_matrix[,4] + 5 * X_matrix[,5] +                   # 线性项
       noise_std * rnorm(n_samples)                           # 噪声
  df_result   <- as.data.frame(X_matrix); names(df_result)   <- paste0("X", 1:n_features); df_result$y   <- y_vector # 组装数据框
  df_result                                              # 返回
}
# ---- 1-SE 剪枝函数就地定义 ----
prune_1se   <- function(model_object, se = 1) {           # 1-SE 剪枝函数
  cptab   <- model_object$cptable                         # 取出 cp 表
  xerror   <- cptab[, "xerror"]                     # 交叉验证误差均值列
  xstd   <- cptab[, "xstd"]                         # 交叉验证标准差列
  cp_values   <- cptab[, "CP"]                             # 复杂度参数列
  min_idx   <- which.min(xerror)                    # xerror 最小处下标
  threshold   <- xerror[min_idx] + se * xstd[min_idx] # 1-SE 阈值
  cp_best   <- max(cp_values[xerror <= threshold])         # 阈值内取最大 cp
  rpart::prune(model_object, cp = cp_best)              # 按 cp 剪枝返回子树
}
set.seed(943); d_fried   <- gen_friedman1(1000, n_features = 7, noise_std = 0.1) # 生成数据
set.seed(100); train_idx_fr   <- sample(nrow(d_fried), nrow(d_fried)*0.5) # 随机抽一半作训练集
tr_fried   <- d_fried[train_idx_fr, ]                                  # 训练集
set.seed(2); m_fried   <- rpart(y ~ ., data = tr_fried, cp = 0) # 建回归树，充分生长
m_fried_best   <- prune_1se(m_fried, se = 1)                # 1-SE 剪枝

# 修复：定义更稳健的预测包装函数，确保参数名称匹配 vip 的内部调用
predict_wrapper   <- function(object, newdata) {
  predict(object, newdata = newdata)
}

vis_perm   <- vi(m_fried_best, method = "permute", train = tr_fried, target = "y", # 置换重要性
          metric = "rmse", pred_wrapper = predict_wrapper, nsim = 10) # 指标、预测函数、重复次数
ggplot(vis_perm, aes(x = reorder(Variable, -Importance), y = Importance)) + # 按重要性排序
  labs(x = "Variable") + geom_bar(stat = "identity") + # 条形图
  geom_errorbar(aes(ymin = Importance - StDev, ymax = Importance + StDev), # 误差棒
                width = 0.5, size = 1)             # 误差棒宽度与线宽 

permutation 法正确地把 X6、X7 排在末位（重要性接近 0）。

回到鲍鱼问题：若要做工艺或测量决策，建议以 permutation importance 的结果为准，它更能反映变量对 泛化预测 的真实贡献。 

 >  本节要点 
 > 1. importance 解读前先勘察数据：确认相关/无关变量的边际信号，作为判断 importance 是否可靠的基准。
 > 2. rpart 自带 importance 基于训练集 improve，对过拟合/代理敏感，可能高估无关变量。
 > 3. permutation importance 基于预测误差，更可靠；两者应结合看。 

## 3.4 时间复杂度 

三个实验（分类-全类别、分类-全连续、回归）合并为一个函数，只换数据源。预期：每层排序 O(p·n log n)，深度受 cp 控制，总时间近线性于 n。 每个实验块先勘察生成数据，再跑时间实验 ；每块自带数据生成与函数定义，独立可运行。三类数据均为模拟生成，目的是隔离「变量类型」对建树耗时的影响。 

以下代码 生成一个纯类别型自变量的分类模拟数据集 ，并进行初步勘察。除了打印摘要，重点查看了目标变量的平衡性以及各个类别变量内部的水平分布情况。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(skimr); library(dplyr)                    # 加载 skimr 与 dplyr
set.seed(2024)                                    # 固定种子
n_samples_cat   <- 5000                                         # 样本量
generate_cat_var   <- function(n) sample(letters[1:5], n, replace = TRUE) # 生成 5 水平类别变量的辅助函数
d_sim_cat   <- data.frame(A = generate_cat_var(n_samples_cat), B = generate_cat_var(n_samples_cat), C = generate_cat_var(n_samples_cat), D = generate_cat_var(n_samples_cat)) # 4 个类别自变量
score_cat   <- (d_sim_cat$A %in% c("a", "b")) +            # 规则 1
         (d_sim_cat$B %in% c("c", "d")) +            # 规则 2
         (d_sim_cat$C %in% c("a", "e")) +            # 规则 3
         (d_sim_cat$D %in% c("b", "c"))              # 规则 4
d_sim_cat$Edibility   <- factor(ifelse(score_cat >= 2, "poisonous", "edible")) # 因变量

d_sim_cat %>% skim_tee()                             # 打印全表摘要并返回原数据
table(d_sim_cat$Edibility)                           # 因变量频数（看两类规模）
prop.table(table(d_sim_cat$Edibility))               # 因变量比例（看是否平衡）
lapply(d_sim_cat[, 1:4], table)                      # 四个类别变量各水平样本量（skim 只给 top_counts） 

以下代码 定义并执行一个时间测试实验（针对全类别型数据） 。通过循环在不同的样本比例下重复建树，记录并绘制出耗时随样本量增长的箱线图和均值折线，以观察 rpart 在处理类别特征时的伸缩性。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
# 分类-全类别
library(rpart)                                    # 加载 rpart 包
# ---- 全类别型分类数据就地生成 ----
set.seed(2024)                                    # 固定种子
n_samples_cat   <- 5000                                         # 样本量
generate_cat_var   <- function(n) sample(letters[1:5], n, replace = TRUE) # 生成 5 水平类别变量的辅助函数
d_sim_cat   <- data.frame(A = generate_cat_var(n_samples_cat), B = generate_cat_var(n_samples_cat), C = generate_cat_var(n_samples_cat), D = generate_cat_var(n_samples_cat)) # 4 个类别自变量
score_cat   <- (d_sim_cat$A %in% c("a", "b")) +            # 规则 1
         (d_sim_cat$B %in% c("c", "d")) +            # 规则 2
         (d_sim_cat$C %in% c("a", "e")) +            # 规则 3
         (d_sim_cat$D %in% c("b", "c"))              # 规则 4
d_sim_cat$Edibility   <- factor(ifelse(score_cat >= 2, "poisonous", "edible")) # 因变量

time_experiment   <- function(data_source, formula_str, N_steps = 10, M_reps = 20) { # 时间实验函数
  total_rows   <- nrow(data_source); result_matrix   <- matrix(NA, M_reps, N_steps)        # 样本数与结果矩阵
  progress_bar   <- txtProgressBar(style = 3)                 # 进度条
  for (step_n in 1:N_steps) {                                # 遍历不同样本比例
    for (rep_m in 1:M_reps) {                              # 每个比例重复 M 次
      set.seed(step_n * 1000 + rep_m)                      # 固定种子
      subset_data   <- data_source[sample(total_rows, step_n * total_rows / N_steps), ]         # 取前 n/N 比例的样本
      start_time   <- Sys.time(); rpart(formula_str, data = subset_data, cp = 0) # 计时建树
      result_matrix[rep_m, step_n]   <- as.numeric(difftime(Sys.time(), start_time, units = "secs")) # 记录耗时
    }
    setTxtProgressBar(progress_bar, step_n / N_steps)                  # 更新进度条
  }
  boxplot(result_matrix, xlab = "每单位=10% 样本", ylab = "time (s)") # 箱线图
  lines(colMeans(result_matrix), type = "o", lwd = 2); grid() # 均值折线
  invisible(result_matrix)                                   # 静默返回结果矩阵
}
time_experiment(d_sim_cat, Edibility ~ .)            # 运行全类别分类实验 

以下代码 生成一个纯连续型自变量的分类模拟数据集 。代码通过生成多个特征的均匀分布，再根据特定规则组合成二分类目标变量，并用  `skim_tee()`  等函数勘察数据的平衡和特征范围。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(skimr); library(dplyr)                    # 加载 skimr 与 dplyr
gen_friedman1   <- function(n_samples, n_features = 10, noise_std = 0.1) { # friedman1 数据生成函数
  X_matrix   <- matrix(runif(n_samples * n_features), n_samples, n_features)               # 生成 [0,1] 均匀自变量
  y_vector   <- 10 * sin(pi * X_matrix[,1] * X_matrix[,2]) +             # 交互项
       20 * (X_matrix[,3] - 0.5)^2 +                     # 二次项
       10 * X_matrix[,4] + 5 * X_matrix[,5] +                   # 线性项
       noise_std * rnorm(n_samples)                           # 噪声
  df_result   <- as.data.frame(X_matrix); names(df_result)   <- paste0("X", 1:n_features); df_result$y   <- y_vector # 组装数据框
  df_result                                              # 返回
}
set.seed(943); d_sim_cont   <- gen_friedman1(1000, n_features = 20, noise_std = 0.1) # 生成 20 个自变量的数据
d_sim_cont$y   <- factor(ifelse(d_sim_cont$y < 14.2, 0, 1))       # 因变量二值化

d_sim_cont %>% skim_tee()                                # 打印全表摘要并返回原数据
table(d_sim_cont$y); prop.table(table(d_sim_cont$y))            # 二值化后平衡性与比例
range(d_sim_cont[, 1:20])                                # 连续自变量范围 

以下代码 执行纯连续型变量的时间测试实验 。它将之前定义的  `time_experiment()`  应用于刚才生成的连续型模拟数据，绘制出耗时随样本量增加的箱线图，便于与类别型数据的测试结果进行横向对比。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
# 分类-全连续
library(rpart)                                    # 加载 rpart 包
# ---- friedman1 数据生成函数就地定义 ----
gen_friedman1   <- function(n_samples, n_features = 10, noise_std = 0.1) { # friedman1 数据生成函数
  X_matrix   <- matrix(runif(n_samples * n_features), n_samples, n_features)               # 生成 [0,1] 均匀自变量
  y_vector   <- 10 * sin(pi * X_matrix[,1] * X_matrix[,2]) +             # 交互项
       20 * (X_matrix[,3] - 0.5)^2 +                     # 二次项
       10 * X_matrix[,4] + 5 * X_matrix[,5] +                   # 线性项
       noise_std * rnorm(n_samples)                           # 噪声
  df_result   <- as.data.frame(X_matrix); names(df_result)   <- paste0("X", 1:n_features); df_result$y   <- y_vector # 组装数据框
  df_result                                              # 返回
}
time_experiment   <- function(data_source, formula_str, N_steps = 10, M_reps = 20) { # 时间实验函数
  total_rows   <- nrow(data_source); result_matrix   <- matrix(NA, M_reps, N_steps)        # 样本数与结果矩阵
  progress_bar   <- txtProgressBar(style = 3)                 # 进度条
  for (step_n in 1:N_steps) {                                # 遍历不同样本比例
    for (rep_m in 1:M_reps) {                              # 每个比例重复 M 次
      set.seed(step_n * 1000 + rep_m)                      # 固定种子
      subset_data   <- data_source[sample(total_rows, step_n * total_rows / N_steps), ]         # 取前 n/N 比例的样本
      start_time   <- Sys.time(); rpart(formula_str, data = subset_data, cp = 0) # 计时建树
      result_matrix[rep_m, step_n]   <- as.numeric(difftime(Sys.time(), start_time, units = "secs")) # 记录耗时
    }
    setTxtProgressBar(progress_bar, step_n / N_steps)                  # 更新进度条
  }
  boxplot(result_matrix, xlab = "每单位=10% 样本", ylab = "time (s)") # 箱线图
  lines(colMeans(result_matrix), type = "o", lwd = 2); grid() # 均值折线
  invisible(result_matrix)                                   # 静默返回结果矩阵
}
set.seed(943); d_sim_cont   <- gen_friedman1(1000, n_features = 20, noise_std = 0.1) # 生成数据
d_sim_cont$y   <- factor(ifelse(d_sim_cont$y < 14.2, 0, 1))       # 因变量二值化
time_experiment(d_sim_cont, y ~ .)                       # 运行全连续分类实验 

以下代码 生成一个回归任务的模拟数据集 ，并进行勘察。不同于前面的分类任务，这里的目标变量 y 保持为连续值，方便后续测试回归树的建树耗时。代码查看了  `skim_tee()`  摘要、目标值的方差以及特征范围。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
library(skimr); library(dplyr)                    # 加载 skimr 与 dplyr
gen_friedman1   <- function(n_samples, n_features = 10, noise_std = 0.1) { # friedman1 数据生成函数
  X_matrix   <- matrix(runif(n_samples * n_features), n_samples, n_features)               # 生成 [0,1] 均匀自变量
  y_vector   <- 10 * sin(pi * X_matrix[,1] * X_matrix[,2]) +             # 交互项
       20 * (X_matrix[,3] - 0.5)^2 +                     # 二次项
       10 * X_matrix[,4] + 5 * X_matrix[,5] +                   # 线性项
       noise_std * rnorm(n_samples)                           # 噪声
  df_result   <- as.data.frame(X_matrix); names(df_result)   <- paste0("X", 1:n_features); df_result$y   <- y_vector # 组装数据框
  df_result                                              # 返回
}
set.seed(943); d_sim_reg   <- gen_friedman1(1000, n_features = 20, noise_std = 0.1) # 生成 20 个自变量的数据

d_sim_reg %>% skim_tee()                                # 打印全表摘要并返回原数据
var(d_sim_reg$y)                                        # 因变量方差（skim 不给方差）
range(d_sim_reg[, 1:20])                                # 连续自变量范围 

以下代码 执行回归任务的时间测试实验 。通过复用之前的  `time_experiment()`  函数，在 20 个连续自变量的回归数据集上进行测试，绘制箱线图以观察在  `cp = 0`  且完全生长的条件下，回归树的建树耗时如何随样本量线性增长。 

In [ ]:
options(warn = -1)                                # 忽略一切警告
options("width" = 140)                            # 充分利用打印宽度
options(repr.plot.width = 22, repr.plot.height = 10) # 满幅
# 回归
library(rpart)                                    # 加载 rpart 包
# ---- friedman1 数据生成函数就地定义 ----
gen_friedman1   <- function(n_samples, n_features = 10, noise_std = 0.1) { # friedman1 数据生成函数
  X_matrix   <- matrix(runif(n_samples * n_features), n_samples, n_features)               # 生成 [0,1] 均匀自变量
  y_vector   <- 10 * sin(pi * X_matrix[,1] * X_matrix[,2]) +             # 交互项
       20 * (X_matrix[,3] - 0.5)^2 +                     # 二次项
       10 * X_matrix[,4] + 5 * X_matrix[,5] +                   # 线性项
       noise_std * rnorm(n_samples)                           # 噪声
  df_result   <- as.data.frame(X_matrix); names(df_result)   <- paste0("X", 1:n_features); df_result$y   <- y_vector # 组装数据框
  df_result                                              # 返回
}
time_experiment   <- function(data_source, formula_str, N_steps = 10, M_reps = 20) { # 时间实验函数
  total_rows   <- nrow(data_source); result_matrix   <- matrix(NA, M_reps, N_steps)        # 样本数与结果矩阵
  progress_bar   <- txtProgressBar(style = 3)                 # 进度条
  for (step_n in 1:N_steps) {                                # 遍历不同样本比例
    for (rep_m in 1:M_reps) {                              # 每个比例重复 M 次
      set.seed(step_n * 1000 + rep_m)                      # 固定种子
      subset_data   <- data_source[sample(total_rows, step_n * total_rows / N_steps), ]         # 取前 n/N 比例的样本
      start_time   <- Sys.time(); rpart(formula_str, data = subset_data, cp = 0) # 计时建树
      result_matrix[rep_m, step_n]   <- as.numeric(difftime(Sys.time(), start_time, units = "secs")) # 记录耗时
    }
    setTxtProgressBar(progress_bar, step_n / N_steps)                  # 更新进度条
  }
  boxplot(result_matrix, xlab = "每单位=10% 样本", ylab = "time (s)") # 箱线图
  lines(colMeans(result_matrix), type = "o", lwd = 2); grid() # 均值折线
  invisible(result_matrix)                                   # 静默返回结果矩阵
}
set.seed(943); d_sim_reg   <- gen_friedman1(1000, n_features = 20, noise_std = 0.1) # 生成数据
time_experiment(d_sim_reg, y ~ .)                       # 运行回归实验 

三条曲线均近线性——rpart 每层排序 O(p·n log n)，树深受 cp 限制，故总时间对 n 近线性。 

 >  本节要点 
 > 1. 时间实验前先勘察生成数据的样本量、变量类型与因变量分布，确认实验条件。
 > 2. rpart 训练时间对样本量近线性；自变量数 p 通过每层排序影响常数。
 > 3. 类别自变量需枚举二分组合，p 大时比连续自变量慢。 

---

## 系列总结

四个文件构成完整的 rpart 教程：

| 文件 | 核心内容 | 附录 |
|---|---|---|
|  `00_总览与环境.ipynb`  | 包定位、函数地图、术语、控制参数、1-SE 定义、环境设置 | A（predict 速查）、B（常见坑） |
|  `01_建树与剪枝机制.ipynb`  | 不纯度、improve、cp、cp 表、1-SE、三棵树对比 |  A（Breiman 排序定理证明） 、 B（1-SE 深探）  |
|  `02_按method展开.ipynb`  | class / anova / poisson / 生存，各 method 特有部分 |  C（其余函数与数据集）  |
|  `03_横切专题.ipynb`  | 代价敏感三法、surrogate、重要性、时间复杂度 | - |

 建议阅读顺序 ：按 0 → 1 → 2 → 3 顺序读；若只想查 API，直接看 0.2 函数地图与附录 A；若只想看某 method，先读 1.4 cp 表读法，再跳到 2 对应小节；若要做代价敏感或缺失值建模，直接看 3.1 / 3.2。

 回到实践 ：rpart 是「可解释、鲁棒、工程友好」的单树基线首选；吃透本系列后，再上随机森林 / GBDT 冲精度，或换  `partykit`  做条件推断树，都会有清晰的对照。 